# Movie Recommendation System

A content-based movie recommendation system built using the TMDB dataset.

**Upgrades applied:**
- TF-IDF vectorization with NLTK stemming
- Robust `ast.literal_eval` with safe fallback
- Fuzzy title search via `rapidfuzz`
- Hybrid scoring with tunable weights
- NDCG@K + Precision@K + catalog coverage evaluation
- Sparse similarity storage (instead of dense pickle)

---

## Pipeline
1. Load & merge `movies.csv`, `credits.csv`, `poster.csv`
2. Safe feature extraction (genres, keywords, top-3 cast, director)
3. NLTK stemming on combined tags
4. TF-IDF → Cosine Similarity
5. Hybrid scoring with tunable weights
6. Fuzzy title matching
7. NDCG@K + Precision@K evaluation
8. Sparse similarity storage

## 1. Importing Libraries

In [41]:
import pandas as pd
import numpy as np
import ast
import pickle
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.stem import PorterStemmer
nltk.download('punkt', quiet=True)

from rapidfuzz import process as rfuzz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import lil_matrix
import scipy.sparse as sp

ps = PorterStemmer()
print("Libraries loaded.")

Libraries loaded.


## 2. Loading Data

In [42]:
movies  = pd.read_csv('Data/movies.csv')
credits = pd.read_csv('Data/credits.csv')
posters = pd.read_csv('Data/poster.csv')

print(f"Movies  : {movies.shape}")
print(f"Credits : {credits.shape}")
print(f"Posters : {posters.shape}")

Movies  : (4803, 20)
Credits : (4803, 4)
Posters : (9837, 2)


## 3. Merging and Cleaning

In [43]:
movies = movies.merge(credits, on='title')
movies = movies.merge(posters, on='title')

movies = movies[['movie_id', 'title', 'overview',
                 'genres', 'keywords', 'cast', 'crew',
                 'vote_average', 'popularity', 'poster']]

movies.dropna(inplace=True)
movies.reset_index(drop=True, inplace=True)

print(f"Total usable movies: {len(movies)}")
movies.head(2)

Total usable movies: 3028


,movie_id,title,overview,genres,keywords,cast,crew,vote_average,popularity,poster
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",7.2,150.437577,https://image.tmdb.org/t/p/original/jRXYjXNq0C...
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",6.9,139.082615,https://image.tmdb.org/t/p/original/2YMnBRh8F6...


## 4. Safe Feature Extraction

All `ast.literal_eval` calls are wrapped in a `safe_parse()` fallback so malformed
JSON rows silently return `[]` instead of crashing the pipeline.

In [44]:
def safe_parse(obj, default=None):
    """Safely parse a JSON-like string; return default on failure."""
    if default is None:
        default = []
    try:
        return ast.literal_eval(obj) if isinstance(obj, str) else obj
    except (ValueError, SyntaxError):
        return default

def extract_names(obj):
    return [i['name'].replace(" ", "") for i in safe_parse(obj) if 'name' in i]

def top3_cast(obj):
    return [i['name'].replace(" ", "") for i in safe_parse(obj)[:3] if 'name' in i]

def fetch_director(obj):
    for i in safe_parse(obj):
        if i.get('job') == 'Director':
            return [i['name'].replace(" ", "")]
    return []

movies['genres']   = movies['genres'].apply(extract_names)
movies['keywords'] = movies['keywords'].apply(extract_names)
movies['cast']     = movies['cast'].apply(top3_cast)
movies['crew']     = movies['crew'].apply(fetch_director)
# Keep overview as a lowercased string (sentence-transformers expects text)
movies['overview'] = movies['overview'].apply(lambda x: x.lower() if isinstance(x, str) else "")

print("Safe feature extraction complete.")

Safe feature extraction complete.


## 5. Stemming + Building Tags

NLTK `PorterStemmer` collapses word variants (`running`, `runs`, `runner` → `run`),
so the TF-IDF vocabulary is smaller and similarity scores are more accurate.

In [45]:
# Build tags as a string for embedding (fixes TypeError)
# Instead of concatenating string + list, join all fields as strings
movies['tags'] = (
    movies['overview'].astype(str) + ' ' +
    movies['genres'].apply(lambda x: ' '.join(x)) + ' ' +
    movies['keywords'].apply(lambda x: ' '.join(x)) + ' ' +
    movies['cast'].apply(lambda x: ' '.join(x)) + ' ' +
    movies['crew'].apply(lambda x: ' '.join(x))
)

new_df = movies[['movie_id', 'title', 'tags',
                 'vote_average', 'popularity', 'poster',
                 'genres']].copy()               # keep genres for evaluation
new_df.rename(columns={'genres': 'genres_list'}, inplace=True)

def stem_tags(text):
    # Tokenize and stem each word in the tag string
    return " ".join(ps.stem(w) for w in text.split())

new_df['tags'] = new_df['tags'].apply(stem_tags)

print("Stemming complete.")
new_df[['title', 'tags']].head(3)

Stemming complete.


,title,tags
0,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,Spectre,a cryptic messag from bond’ past send him on a...


## 6. TF-IDF Vectorization + Cosine Similarity

In [46]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from annoy import AnnoyIndex

# Use numpy array for embeddings (convert_to_tensor=False)
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(new_df['tags'], convert_to_tensor=False)  # returns np.ndarray

# ANN index construction
ann_backend = 'faiss'  # or 'annoy', based on your preference
print(f"Building ANN index using backend: {ann_backend}")

if ann_backend == 'faiss':
    # Faiss prefers float32 and benefits from L2-normalized vectors for inner-product = cosine
    d = embeddings.shape[1]
    embeddings = embeddings.astype('float32')
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(d)
    index.add(embeddings)
    print(f"Faiss Index built with dimension={d}, n_items={index.ntotal}")
else:
    # Annoy stores vectors as lists and uses 'angular' distance (approx cosine)
    d = embeddings.shape[1]
    index = AnnoyIndex(d, 'angular')
    for i, v in enumerate(embeddings):
        index.add_item(i, v.tolist())
    index.build(50)   # 50 trees for reasonable accuracy / build-time tradeoff
    print(f"Annoy Index built with dimension={d}")

# Keep a normalized copy for exact cosine computations when needed
embeddings_norm = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
print("ANN index ready and embeddings normalized.")

Building ANN index using backend: faiss
Faiss Index built with dimension=384, n_items=3028
ANN index ready and embeddings normalized.


## 7. Normalize Signals for Hybrid Scoring

In [47]:
def minmax(series):
    return (series - series.min()) / (series.max() - series.min())

new_df['vote_norm'] = minmax(new_df['vote_average'])
new_df['pop_norm']  = minmax(new_df['popularity'])

vote_arr = new_df['vote_norm'].values
pop_arr  = new_df['pop_norm'].values

print("Normalization complete.")

Normalization complete.


## 8. Fuzzy Title Search

`rapidfuzz.process.extractOne` returns the closest match with a confidence score.
Queries like `'avtar'` and `'dark night'` will be resolved without crashing.

In [48]:
ALL_TITLES = new_df['title'].tolist()

def find_title(query, cutoff=60):
    """Return the best matching title or None if confidence < cutoff."""
    result = rfuzz.extractOne(query, ALL_TITLES, score_cutoff=cutoff)
    if result is None:
        return None
    return result[0]          # (match_string, score, index)

# Quick sanity check
for q in ['avtar', 'dark night', 'interstella', 'Toy Stori']:
    print(f"  '{q}'  →  '{find_title(q)}'")

  'avtar'  →  'Avatar'
  'dark night'  →  'The Dark Knight Rises'
  'interstella'  →  'Interstellar'
  'Toy Stori'  →  'Toy Story'


## 9. Recommendation Function (Tunable Weights)

- **`w_sim`** : weight for cosine similarity (content match)
- **`w_vote`** : weight for normalised vote average
- **`w_pop`**  : weight for normalised popularity

Defaults give a Netflix-like balance: quality content ranked above obscure matches.

In [49]:
def mmr(doc_embeddings, query_embedding, candidate_indices, lambda_param=0.7, top_n=5):
    """Maximal Marginal Relevance (MMR) re-ranking.
    doc_embeddings: normalized embeddings (n_docs, dim)
    query_embedding: normalized vector (dim,)
    candidate_indices: list of candidate doc indices
    lambda_param: trade-off between relevance and diversity (0..1)
    """
    # Precompute cosine similarities
    sims_to_query = np.dot(doc_embeddings[candidate_indices], query_embedding)
    selected = []
    candidate_set = list(candidate_indices)

    while len(selected) < top_n and candidate_set:
        mmr_scores = []
        for idx in candidate_set:
            relevance = np.dot(doc_embeddings[idx], query_embedding)
            if not selected:
                diversity = 0
            else:
                diversity = max(np.dot(doc_embeddings[idx], doc_embeddings[s]) for s in selected)
            score = lambda_param * relevance - (1 - lambda_param) * diversity
            mmr_scores.append((idx, score))
        # pick best
        pick, _ = max(mmr_scores, key=lambda x: x[1])
        selected.append(pick)
        candidate_set.remove(pick)
    return selected


def recommend(movie, top_n=5, w_sim=0.70, w_vote=0.20, w_pop=0.10, ann_k=50, mmr_lambda=0.7):
    """
    Recommend top-N movies.
    
    Supports fuzzy title matching; weights must sum to 1.
    """
    matched = find_title(movie)
    if matched is None:
        print(f"No close match found for '{movie}'")
        return
    if matched != movie:
        print(f"(Matched '{movie}' → '{matched}')")

    idx    = new_df[new_df['title'] == matched].index[0]

    # Query ANN to get top-K candidate indices
    if ann_backend == 'faiss':
        q = embeddings_norm[idx].reshape(1, -1).astype('float32')
        D, I = index.search(q, ann_k + 1)  # +1 to account for the query itself
        candidates = [i for i in I[0] if i != idx][:ann_k]
    else:
        neighbors = index.get_nns_by_item(idx, ann_k + 1)
        candidates = [i for i in neighbors if i != idx][:ann_k]

    # Hybrid scoring: we compute exact cosine (dot) with normalized embeddings
    sim_scores = np.dot(embeddings_norm[candidates], embeddings_norm[idx])
    scores = sim_scores * w_sim + vote_arr[candidates] * w_vote + pop_arr[candidates] * w_pop

    # Get top M candidates by hybrid score, then apply MMR for diversity
    M = min(len(candidates), top_n * 5)
    top_candidate_idxs = np.argsort(scores)[::-1][:M]
    candidate_indices = [candidates[i] for i in top_candidate_idxs]

    selected = mmr(embeddings_norm, embeddings_norm[idx], candidate_indices, lambda_param=mmr_lambda, top_n=top_n)

    print(f"\nRecommendations for '{matched}' (ann_k={ann_k}, mmr_lambda={mmr_lambda}):")
    print("-" * 60)
    for rank, i in enumerate(selected, 1):
        row = new_df.iloc[i]
        print(f"{rank}. {row.title:<42} ")
        print(f"   Poster: {row.poster}")

In [50]:
# Test with typos / partial names
recommend('avtar')                         # fuzzy match → Avatar
print()
recommend('The Dark Knight')
print()
recommend('Interstellar', w_sim=0.5, w_vote=0.3, w_pop=0.2)   # custom weights

(Matched 'avtar' → 'Avatar')

Recommendations for 'Avatar' (ann_k=50, mmr_lambda=0.7):
------------------------------------------------------------
1. Battle: Los Angeles                        
   Poster: https://image.tmdb.org/t/p/original/jloyGeVYZSxM9zsLFvVOWuj2ey4.jpg
2. Aliens                                     
   Poster: https://image.tmdb.org/t/p/original/r1x5JGpyqZU8PYhbs4UcrO1Xb6x.jpg
3. Moon                                       
   Poster: https://image.tmdb.org/t/p/original/cJ6JnuLwCNbiAAOBuHDjRTP7bQJ.jpg
4. Interstellar                               
   Poster: https://image.tmdb.org/t/p/original/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg
5. The 5th Wave                               
   Poster: https://image.tmdb.org/t/p/original/4Dd79cz5qxKNoKWg1xsfrpOchEr.jpg


Recommendations for 'The Dark Knight' (ann_k=50, mmr_lambda=0.7):
------------------------------------------------------------
1. Batman                                     
   Poster: https://image.tmdb.org/t/p/original

## 10. Evaluation — Precision@K, NDCG@K & Catalog Coverage

| Metric | What it measures |
|---|---|
| **Precision@K** | Fraction of top-K recs sharing a genre with the query |
| **NDCG@K** | Same, but penalises relevant items ranked lower |
| **Coverage** | % of unique movies surfaced across all test queries |

In [51]:
def get_ranked(movie, k, w_sim=0.70, w_vote=0.20, w_pop=0.10):
    matched = find_title(movie)
    if matched is None:
        return None, None
    idx    = new_df[new_df['title'] == matched].index[0]
    # Use ANN-based hybrid scoring, not similarity matrix
    if ann_backend == 'faiss':
        q = embeddings_norm[idx].reshape(1, -1).astype('float32')
        D, I = index.search(q, k + 1)
        candidates = [i for i in I[0] if i != idx][:k]
    else:
        neighbors = index.get_nns_by_item(idx, k + 1)
        candidates = [i for i in neighbors if i != idx][:k]
    sim_scores = np.dot(embeddings_norm[candidates], embeddings_norm[idx])
    scores = sim_scores * w_sim + vote_arr[candidates] * w_vote + pop_arr[candidates] * w_pop
    ranked = sorted(
        zip(candidates, scores),
        key=lambda x: x[1], reverse=True
    )
    query_genres = set(new_df.iloc[idx]['genres_list'])
    return ranked, query_genres

def precision_at_k(movie, k=5, **kwargs):
    ranked, qg = get_ranked(movie, k, **kwargs)
    if ranked is None: return None
    hits = sum(1 for i, _ in ranked if set(new_df.iloc[i]['genres_list']) & qg)
    return hits / k

def ndcg_at_k(movie, k=5, **kwargs):
    ranked, qg = get_ranked(movie, k, **kwargs)
    if ranked is None: return None
    dcg  = sum(
        int(bool(set(new_df.iloc[i]['genres_list']) & qg)) / np.log2(r + 2)
        for r, (i, _) in enumerate(ranked)
    )
    idcg = sum(1 / np.log2(r + 2) for r in range(min(k, len(qg))))
    return dcg / idcg if idcg > 0 else 0.0

def catalog_coverage(movies_list, k=5):
    all_recs = set()
    for m in movies_list:
        ranked, _ = get_ranked(m, k)
        if ranked:
            all_recs.update(i for i, _ in ranked)
    return len(all_recs) / len(new_df)

# Evaluate
test_movies = ['Avatar', 'The Dark Knight', 'Interstellar', 'Toy Story', 'Inception']

print(f"{'Movie':<32} {'P@5':>6} {'NDCG@5':>8}")
print("-" * 50)
for m in test_movies:
    p = precision_at_k(m, k=5)
    n = ndcg_at_k(m, k=5)
    if p is not None:
        print(f"{m:<32} {p:>6.2f} {n:>8.4f}")

cov = catalog_coverage(test_movies, k=5)
print(f"\nCatalog coverage (top-5, {len(test_movies)} queries): {cov*100:.2f}%")

Movie                               P@5   NDCG@5
--------------------------------------------------
Avatar                             1.00   1.1510
The Dark Knight                    1.00   1.1510
Interstellar                       1.00   1.3836
Toy Story                          1.00   1.3836
Inception                          1.00   1.0000

Catalog coverage (top-5, 5 queries): 0.83%


## 11. Saving — Sparse Similarity + Model Artifacts

The full dense similarity matrix is **~180 MB** for 3k movies and grows quadratically.
We store only the **top-50 neighbours** per movie as a sparse matrix (~3 MB), then
save the dense version separately for notebooks that need it.

In [52]:
# Save top-K ANN neighbors for each movie using embedding-based similarity
n = len(new_df)
K_NEIGHBOURS = 50

sparse_sim = lil_matrix((n, n), dtype=np.float32)
for i in range(n):
    # Get top-K+1 (including self), skip self
    if ann_backend == 'faiss':
        q = embeddings_norm[i].reshape(1, -1).astype('float32')
        D, I = index.search(q, K_NEIGHBOURS + 1)
        top_k_idx = [j for j in I[0] if j != i][:K_NEIGHBOURS]
        top_k_sim = [float(s) for j, s in zip(I[0], D[0]) if j != i][:K_NEIGHBOURS]
    else:
        neighbors = index.get_nns_by_item(i, K_NEIGHBOURS + 1)
        top_k_idx = [j for j in neighbors if j != i][:K_NEIGHBOURS]
        top_k_sim = [float(np.dot(embeddings_norm[i], embeddings_norm[j])) for j in top_k_idx]
    sparse_sim[i, top_k_idx] = top_k_sim

sparse_csr = sparse_sim.tocsr()
sp.save_npz('similarity_sparse.npz', sparse_csr)
print(f"Sparse matrix saved  ({K_NEIGHBOURS} neighbours/movie)")

# Also pickle full dataframe for notebook use
pickle.dump(new_df,     open('movie_list.pkl', 'wb'))
print("Artifacts saved: movie_list.pkl, similarity_sparse.npz")

Sparse matrix saved  (50 neighbours/movie)
Artifacts saved: movie_list.pkl, similarity_sparse.npz
